# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashizhenya755-dev/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


For my lane (Refresh / Content Opportunity Scoring), one row in
fact_content_daily_performance means one content page, for one client, on
one specific day (grain: report_date + client_hash_id + content_hash_id).
This is different from the starter CSV, where one row was a page's entire
90-day summary — here the warehouse gives me a full daily time series per
page.

Table: fact_content_daily_performance, joined to dim_content and
dim_clients as needed.

Time window for this notebook: month=2026-03, a mid-panel month, so I'm
not developing label logic on the sealed final month (June 2026).

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install -q huggingface_hub duckdb
month_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

schema = con.sql(f"DESCRIBE SELECT * FROM '{month_path}'").df()
print(schema)

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [34]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [35]:
import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
print('DuckDB connected to Hugging Face')

DuckDB connected to Hugging Face


In [36]:
test = con.sql("SELECT COUNT(*) AS n FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'").df()
print(test)

     n
0  104


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (safe to use as model input, observed before the decision point):
gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, scroll_events.
These are raw observed daily measurements from search and analytics.

Label / proxy: I will build a decline label from gsc_impressions or
ga4_sessions trending down over a forward window (e.g. next 30 days),
comparing it against a prior 90-day feature window. This is stronger than
week 2's same-window proxy because it's a genuine past-to-future split.

Context (grouping/joins, not model features): report_date, client_hash_id,
content_hash_id, month.

Excluded: client_has_gsc, client_has_ga4, gsc_data_available,
ga4_data_available. Why: these describe whether tracking existed that day,
not page performance — using them as model features would let the model
learn "was this client even being tracked" instead of real content signal.
I use them only to filter for valid rows, not as inputs.

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample = con.sql(f"SELECT * FROM '{month_path}' LIMIT 5").df()
print(sample.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks on month=2026-03:
1. Grain check: confirm report_date + client_hash_id + content_hash_id has
   no duplicate rows, proving one row really is one page/one client/one day.
2. Row count and date span: how many rows this month covers and which
   dates they span.
3. Availability: how many rows have gsc_data_available IS TRUE, since rows
   before a client's tracking start only carry partial/no data.

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS distinct_grain
    FROM '{month_path}'
""").df()
print("Grain check (total_rows should equal distinct_grain):")
print(grain_check)

span_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM '{month_path}'
""").df()
print("\nRow count and date span:")
print(span_check)

availability_check = con.sql(f"""
    SELECT COUNT(*) AS rows_with_gsc_available
    FROM '{month_path}'
    WHERE gsc_data_available IS TRUE
""").df()
print("\nRows with gsc_data_available IS TRUE:")
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (total_rows should equal distinct_grain):
   total_rows  distinct_grain
0     9841378         9841378

Row count and date span:
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows with gsc_data_available IS TRUE:
   rows_with_gsc_available
0                  3611061


In [39]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        scroll_events
    FROM '{month_path}'
    WHERE gsc_data_available IS TRUE
    LIMIT 1000
""").df()

print(features.shape)
features.head()

(1000, 8)


,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,<NA>,<NA>


Five features, each knowable at the decision moment because it is a raw
observed measurement recorded on or before that day, not derived from a
future outcome:

1. gsc_impressions — knowable because it's logged the same day Search
   Console records the page appearing in results.
2. gsc_clicks — knowable same-day, a direct count of clicks that day.
3. gsc_avg_position — knowable same-day, the page's average search rank
   that day.
4. ga4_sessions — knowable same-day, sessions Analytics recorded that day.
5. scroll_events — knowable same-day, on-page scroll events recorded that
   day.

None of these use tomorrow's data to describe today.

Note on labels: my main contract above describes the real intended label
for this lane — future decline, comparing a prior 90-day window to a
forward 30-day window. For this leakage demonstration specifically, I use
a simpler same-day proxy label, zero_click_day (did this page get zero
clicks that day), only because it's fast to build and easy to leak on
purpose. This is not my final Refresh label — it exists here only to
demonstrate how a label-derived feature causes leakage.

In [40]:
labeled = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        scroll_events,
        CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END AS zero_click_day
    FROM '{month_path}'
    WHERE gsc_data_available IS TRUE
    LIMIT 5000
""").df()

# THE TRAP: a feature built directly from the label itself.
# gsc_clicks_times_10 is clearly derived from gsc_clicks, which is what
# zero_click_day (our label) is calculated from. This is leakage on purpose.
labeled["gsc_clicks_times_10"] = labeled["gsc_clicks"] * 10

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_leaky = labeled[["gsc_impressions", "gsc_avg_position", "gsc_clicks_times_10"]].fillna(0)
y = labeled["zero_click_day"]

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaky_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"WITH the leaked feature (gsc_clicks_times_10), AUC: {leaky_score:.3f}")

# Now remove the leaked feature and re-check honestly.
X_honest = labeled[["gsc_impressions", "gsc_avg_position"]].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print(f"WITHOUT the leaked feature, honest AUC: {honest_score:.3f}")

WITH the leaked feature (gsc_clicks_times_10), AUC: 1.000
WITHOUT the leaked feature, honest AUC: 0.866


The leak: I added gsc_clicks_times_10, a feature built directly from
gsc_clicks — the same column my label (zero_click_day) is calculated
from. This pushed AUC to a perfect 1.000, which should be a red flag, not
a win: a model can't be that good on a real-world signal like clicks
unless it's cheating by seeing the answer.

After removing the leaked feature, the honest AUC drops to 0.866 — lower,
but trustworthy, since it only uses features (impressions, average
position) that don't encode the label itself.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation of this slice: only about 37% of rows in March 2026 have
gsc_data_available IS TRUE (3,611,061 of 9,841,378). This means most of
this month's rows come from clients whose tracking hadn't started yet, or
from days without real search data. Any feature or label I build from GSC
signals must filter on gsc_data_available IS TRUE first, or I'll be
treating "no tracking yet" as if it were "zero traffic" — a mistake that
would badly distort any decline/growth label.

Other limits this slice can't resolve: it's still observational, not
causal (a correlation between a signal and a later outcome doesn't prove a
refresh caused a recovery), and this is one month of one client mix, so
patterns here may not generalize to other months or the full client base
without checking again.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.